# Edge Cases

Runs tiny pathological models and audits experimental, partial, and remaining namespace surface.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_grads=True,
    save_arg_values=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].out.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.ops.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.ops.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.trace(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.trace(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "Trace": log,
    "LayerLog": layer_log,
    "OpLog": layer_pass,
    "ModuleLog": module_log,
    "ModuleCallLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnCallLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer ops, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## Edge Cases: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.capture.arg_positions",
    "tl.capture.flops",
    "tl.capture.output_tensors",
    "tl.capture.salient_args",
    "tl.capture.source_tensors",
    "tl.capture.tensor_tracking",
    "tl.decoration.clear_patch_detached_references_cache",
    "tl.decoration.decorate_all_once",
    "tl.decoration.patch_detached_references",
    "tl.decoration.patch_model_instance",
    "tl.decoration.unwrap_torch",
    "tl.decoration.wrap_torch",
    "tl.decoration.wrapped",
    "tl.examples.load",
    "tl.experimental.AutoCaptureSession",
    "tl.experimental.Session",
    "tl.experimental.attribute_walk",
    "tl.experimental.auto_capture",
    "tl.experimental.dagua",
    "tl.experimental.freeze_module",
    "tl.experimental.node_styles",
    "tl.experimental.session",
    "tl.experimental.stop_after",
    "tl.partial.PartialTrace",
    "tl.partial.from_failed_capture",
    "tl.postprocess.List",
    "tl.postprocess.TYPE_CHECKING",
    "tl.postprocess.ast_branches",
    "tl.postprocess.compute_graph_shape_hash",
    "tl.postprocess.control_flow",
    "tl.postprocess.finalization",
    "tl.postprocess.graph_traversal",
    "tl.postprocess.labeling",
    "tl.postprocess.loop_detection",
    "tl.postprocess.postprocess",
    "tl.postprocess.postprocess_fast",
    "tl.postprocess.safe_copy",
    "tl.postprocess.time",
    "tl.postprocess.torch",
    "tl.utils.AutocastRestore",
    "tl.utils.DoctorCheck",
    "tl.utils.DoctorReport",
    "tl.utils.MAX_FLOATING_POINT_TOLERANCE",
    "tl.utils._ATTR_SKIP_SET",
    "tl.utils._AUTOCAST_DEVICES",
    "tl.utils._cuda_available",
    "tl.utils._get_code_context",
    "tl.utils._is_cuda_available",
    "tl.utils._model_expects_single_arg",
    "tl.utils._safe_copy_arg",
    "tl.utils.assign_to_sequence_or_dict",
    "tl.utils.doctor",
    "tl.utils.ensure_iterable",
    "tl.utils.find_executable_save_set",
    "tl.utils.flop_count",
    "tl.utils.format_flops",
    "tl.utils.format_size",
    "tl.utils.get_attr_values_from_tensor_list",
    "tl.utils.get_memory_amount",
    "tl.utils.get_vars_of_type_from_obj",
    "tl.utils.human_readable_size",
    "tl.utils.identity",
    "tl.utils.in_notebook",
    "tl.utils.index_nested",
    "tl.utils.int_list_to_compact_str",
    "tl.utils.is_iterable",
    "tl.utils.iter_accessible_attributes",
    "tl.utils.list_modules",
    "tl.utils.list_ops",
    "tl.utils.log_current_autocast_state",
    "tl.utils.log_current_rng_states",
    "tl.utils.trace_streaming",
    "tl.utils.make_random_barcode",
    "tl.utils.make_short_barcode_from_input",
    "tl.utils.nested_assign",
    "tl.utils.nested_getattr",
    "tl.utils.normalize_input_args",
    "tl.utils.peek_graph",
    "tl.utils.print_override",
    "tl.utils.progress_bar",
    "tl.utils.register_op_rule",
    "tl.utils.remove_attributes_with_prefix",
    "tl.utils.remove_entry_from_list",
    "tl.utils.safe_copy",
    "tl.utils.safe_copy_args",
    "tl.utils.safe_copy_kwargs",
    "tl.utils.safe_to",
    "tl.utils.set_random_seed",
    "tl.utils.set_rng_from_saved_states",
    "tl.utils.synthetic_input",
    "tl.utils.tensor_all_nan",
    "tl.utils.tensor_nanequal",
    "tl.utils.tensor_stats_summary",
    "tl.utils.warn_parallel",
]

empty = nn.Identity().eval()
single = nn.Sequential(nn.ReLU()).eval()
for label, edge_model, edge_x in [
    ("empty", empty, torch.randn(1, 4)),
    ("single", single, torch.randn(1, 4)),
    ("dynamic", tiny_dynamic_model(seed=9).eval(), torch.randn(1, 4)),
    ("branched", tiny_branched_model(seed=9).eval(), torch.randn(1, 4)),
]:
    try:
        edge_log = tl.trace(edge_model, edge_x, vis_opt="none")
        print(label, len(edge_log.layer_list))
    except Exception as exc:
        print("# skipped", label, type(exc).__name__, str(exc).splitlines()[0][:80])
print("experimental", [name for name in dir(tl.experimental) if not name.startswith("_")][:8])
print("partial", [name for name in dir(tl.partial) if not name.startswith("_")][:8])

## Edge Cases coverage cluster 1

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.capture.arg_positions",
    "tl.capture.flops",
    "tl.capture.output_tensors",
    "tl.capture.salient_args",
    "tl.capture.source_tensors",
    "tl.capture.tensor_tracking",
    "tl.decoration.clear_patch_detached_references_cache",
    "tl.decoration.decorate_all_once",
    "tl.decoration.patch_detached_references",
    "tl.decoration.patch_model_instance",
    "tl.decoration.unwrap_torch",
    "tl.decoration.wrap_torch",
    "tl.decoration.wrapped",
    "tl.examples.load",
    "tl.experimental.AutoCaptureSession",
    "tl.experimental.Session",
    "tl.experimental.attribute_walk",
    "tl.experimental.auto_capture",
    "tl.experimental.dagua",
    "tl.experimental.freeze_module",
    "tl.experimental.node_styles",
    "tl.experimental.session",
    "tl.experimental.stop_after",
    "tl.partial.PartialTrace",
    "tl.partial.from_failed_capture",
    "tl.postprocess.List",
    "tl.postprocess.TYPE_CHECKING",
    "tl.postprocess.ast_branches",
    "tl.postprocess.compute_graph_shape_hash",
    "tl.postprocess.control_flow",
    "tl.postprocess.finalization",
    "tl.postprocess.graph_traversal",
]

audit_touch_items("Edge Cases coverage cluster 1", ITEMS, AUDIT_CONTEXT)

## Edge Cases coverage cluster 2

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.postprocess.labeling",
    "tl.postprocess.loop_detection",
    "tl.postprocess.postprocess",
    "tl.postprocess.postprocess_fast",
    "tl.postprocess.safe_copy",
    "tl.postprocess.time",
    "tl.postprocess.torch",
    "tl.utils.AutocastRestore",
    "tl.utils.DoctorCheck",
    "tl.utils.DoctorReport",
    "tl.utils.MAX_FLOATING_POINT_TOLERANCE",
    "tl.utils._ATTR_SKIP_SET",
    "tl.utils._AUTOCAST_DEVICES",
    "tl.utils._cuda_available",
    "tl.utils._get_code_context",
    "tl.utils._is_cuda_available",
    "tl.utils._model_expects_single_arg",
    "tl.utils._safe_copy_arg",
    "tl.utils.assign_to_sequence_or_dict",
    "tl.utils.doctor",
    "tl.utils.ensure_iterable",
    "tl.utils.find_executable_save_set",
    "tl.utils.flop_count",
    "tl.utils.format_flops",
    "tl.utils.format_size",
    "tl.utils.get_attr_values_from_tensor_list",
    "tl.utils.get_memory_amount",
    "tl.utils.get_vars_of_type_from_obj",
    "tl.utils.human_readable_size",
    "tl.utils.identity",
    "tl.utils.in_notebook",
    "tl.utils.index_nested",
]

audit_touch_items("Edge Cases coverage cluster 2", ITEMS, AUDIT_CONTEXT)

## Edge Cases coverage cluster 3

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.utils.int_list_to_compact_str",
    "tl.utils.is_iterable",
    "tl.utils.iter_accessible_attributes",
    "tl.utils.list_modules",
    "tl.utils.list_ops",
    "tl.utils.log_current_autocast_state",
    "tl.utils.log_current_rng_states",
    "tl.utils.trace_streaming",
    "tl.utils.make_random_barcode",
    "tl.utils.make_short_barcode_from_input",
    "tl.utils.nested_assign",
    "tl.utils.nested_getattr",
    "tl.utils.normalize_input_args",
    "tl.utils.peek_graph",
    "tl.utils.print_override",
    "tl.utils.progress_bar",
    "tl.utils.register_op_rule",
    "tl.utils.remove_attributes_with_prefix",
    "tl.utils.remove_entry_from_list",
    "tl.utils.safe_copy",
    "tl.utils.safe_copy_args",
    "tl.utils.safe_copy_kwargs",
    "tl.utils.safe_to",
    "tl.utils.set_random_seed",
    "tl.utils.set_rng_from_saved_states",
    "tl.utils.synthetic_input",
    "tl.utils.tensor_all_nan",
    "tl.utils.tensor_nanequal",
    "tl.utils.tensor_stats_summary",
    "tl.utils.warn_parallel",
]

audit_touch_items("Edge Cases coverage cluster 3", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.capture.arg_positions",
    "tl.capture.flops",
    "tl.capture.output_tensors",
    "tl.capture.salient_args",
    "tl.capture.source_tensors",
    "tl.capture.tensor_tracking",
    "tl.decoration.clear_patch_detached_references_cache",
    "tl.decoration.decorate_all_once",
    "tl.decoration.patch_detached_references",
    "tl.decoration.patch_model_instance",
    "tl.decoration.unwrap_torch",
    "tl.decoration.wrap_torch",
    "tl.decoration.wrapped",
    "tl.examples.load",
    "tl.experimental.AutoCaptureSession",
    "tl.experimental.Session",
    "tl.experimental.attribute_walk",
    "tl.experimental.auto_capture",
    "tl.experimental.dagua",
    "tl.experimental.freeze_module",
    "tl.experimental.node_styles",
    "tl.experimental.session",
    "tl.experimental.stop_after",
    "tl.partial.PartialTrace",
    "tl.partial.from_failed_capture",
    "tl.postprocess.List",
    "tl.postprocess.TYPE_CHECKING",
    "tl.postprocess.ast_branches",
    "tl.postprocess.compute_graph_shape_hash",
    "tl.postprocess.control_flow",
    "tl.postprocess.finalization",
    "tl.postprocess.graph_traversal",
    "tl.postprocess.labeling",
    "tl.postprocess.loop_detection",
    "tl.postprocess.postprocess",
    "tl.postprocess.postprocess_fast",
    "tl.postprocess.safe_copy",
    "tl.postprocess.time",
    "tl.postprocess.torch",
    "tl.utils.AutocastRestore",
    "tl.utils.DoctorCheck",
    "tl.utils.DoctorReport",
    "tl.utils.MAX_FLOATING_POINT_TOLERANCE",
    "tl.utils._ATTR_SKIP_SET",
    "tl.utils._AUTOCAST_DEVICES",
    "tl.utils._cuda_available",
    "tl.utils._get_code_context",
    "tl.utils._is_cuda_available",
    "tl.utils._model_expects_single_arg",
    "tl.utils._safe_copy_arg",
    "tl.utils.assign_to_sequence_or_dict",
    "tl.utils.doctor",
    "tl.utils.ensure_iterable",
    "tl.utils.find_executable_save_set",
    "tl.utils.flop_count",
    "tl.utils.format_flops",
    "tl.utils.format_size",
    "tl.utils.get_attr_values_from_tensor_list",
    "tl.utils.get_memory_amount",
    "tl.utils.get_vars_of_type_from_obj",
    "tl.utils.human_readable_size",
    "tl.utils.identity",
    "tl.utils.in_notebook",
    "tl.utils.index_nested",
    "tl.utils.int_list_to_compact_str",
    "tl.utils.is_iterable",
    "tl.utils.iter_accessible_attributes",
    "tl.utils.list_modules",
    "tl.utils.list_ops",
    "tl.utils.log_current_autocast_state",
    "tl.utils.log_current_rng_states",
    "tl.utils.trace_streaming",
    "tl.utils.make_random_barcode",
    "tl.utils.make_short_barcode_from_input",
    "tl.utils.nested_assign",
    "tl.utils.nested_getattr",
    "tl.utils.normalize_input_args",
    "tl.utils.peek_graph",
    "tl.utils.print_override",
    "tl.utils.progress_bar",
    "tl.utils.register_op_rule",
    "tl.utils.remove_attributes_with_prefix",
    "tl.utils.remove_entry_from_list",
    "tl.utils.safe_copy",
    "tl.utils.safe_copy_args",
    "tl.utils.safe_copy_kwargs",
    "tl.utils.safe_to",
    "tl.utils.set_random_seed",
    "tl.utils.set_rng_from_saved_states",
    "tl.utils.synthetic_input",
    "tl.utils.tensor_all_nan",
    "tl.utils.tensor_nanequal",
    "tl.utils.tensor_stats_summary",
    "tl.utils.warn_parallel",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")